# FVCOM Surface Forcing File Generation Tutorial

This tutorial demonstrates how to create FVCOM surface forcing files using PyFVCOM2. We'll work through the complete workflow from downloading ERA5 surface forcing to generating the final forcing file.

## Overview
FVCOM grids can take in surface forcing to drive the heating and wind stress in the upper layers of the model. This tutorial shows how to:

1. Retrieve the correct data from ERA5 climate data store
2. Create FVCOM grid objects
3. Read and interpolate the ERA5 data
4. Generate surface forcing files
5. Optionally add ice forcing

## Prerequisites

- PyFVCOM2 installed with all dependencies
- cdsapi setup, or pre downloaded ERA5 data (skip first section if so
- FVCOM grid files (grid definition, open boundary, sigma levels) or restart file
- ice thickness and ice concentraion (e.g. from CMEMS model) for adding ice forcing

## 1. Download ERA5 data

First, we download the appropriate variables from the climate data store for the atmospheric model re-analysis. This loops over the variables required to reduce the size of the download and then combines them into a single netcdf file at the end. This step can be skipped if the correct variables have already been downloaded to cover the FVCOM grid area.

In [1]:
import cdsapi
import numpy as np
import xarray as xr
import os
import tempfile

def get_era5(lon_min, lon_max, lat_min, lat_max, year, month_start, month_end, variables, output_dir):
    '''
    Download ERA5 data using the cdsapi, one request per variable, then merge
    all variables into a single output NetCDF file.

    Each variable is downloaded to a temporary NetCDF file in a temp directory,
    opened with xarray, and merged into a combined dataset. The temporary files
    are deleted once the merge is complete (or if an error occurs).
    '''

    dataset = "reanalysis-era5-single-levels"
    client = cdsapi.Client()

    os.makedirs(output_dir, exist_ok=True)
    merged_filename = f'era5_{year}_{month_start:02d}_{month_end:02d}.nc'
    merged_path = os.path.join(output_dir, merged_filename)

    with tempfile.TemporaryDirectory(dir=output_dir) as tmp_dir:
        temp_paths = []

        for variable in variables:
            request = {
                "product_type": ["reanalysis"],
                "variable": variable,
                "year": year,
                "month": list(range(month_start, month_end+1)),
                "day": [
                    "01", "02", "03",
                    "04", "05", "06",
                    "07", "08", "09",
                    "10", "11", "12",
                    "13", "14", "15",
                    "16", "17", "18",
                    "19", "20", "21",
                    "22", "23", "24",
                    "25", "26", "27",
                    "28", "29", "30",
                    "31"
                ],
                "time": [
                    "00:00", "01:00", "02:00",
                    "03:00", "04:00", "05:00",
                    "06:00", "07:00", "08:00",
                    "09:00", "10:00", "11:00",
                    "12:00", "13:00", "14:00",
                    "15:00", "16:00", "17:00",
                    "18:00", "19:00", "20:00",
                    "21:00", "22:00", "23:00"
                ],
                "data_format": "netcdf",
                "area": [lat_max, lon_min, lat_min, lon_max]
            }

            temp_path = os.path.join(tmp_dir, f'era5_{year}_{month_start:02d}_{month_end:02d}_{variable}.nc')
            client.retrieve(dataset, request).download(temp_path)
            temp_paths.append(temp_path)

        print(f"Merging {len(temp_paths)} variable files into {merged_path} ...")
        # Open each file individually and merge (avoids requiring dask, which
        # open_mfdataset depends on). compat="override" + join="override" keeps
        # this robust to tiny floating-point differences in shared coordinates
        # across separate per-variable downloads.
        datasets = [xr.open_dataset(p) for p in temp_paths]
        try:
            merged_ds = xr.merge(datasets, compat="override", join="override")
            merged_ds.to_netcdf(merged_path)
            merged_ds.close()
        finally:
            for ds in datasets:
                ds.close()

all_variables = [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_temperature",
        "surface_pressure",
        "total_precipitation",
        "mean_surface_downward_long_wave_radiation_flux",
        "mean_surface_net_short_wave_radiation_flux",
        "total_cloud_cover",
        "evaporation",
        "2m_dewpoint_temperature",
        "land_sea_mask"
    ]
year = 2020
month_start = 6
month_end = 12
lon_min = 6.5
lon_max = 18
lat_min = 75
lat_max = 80
get_era5(lon_min, lon_max, lat_min, lat_max, year,
                month_start, month_end, all_variables, output_dir='./')

## 2. Configuration and Setup

Define the paths to the data, get the FVCOM grid object, and define the time period to create the forcing for

In [2]:
import pyfvcom2 as pf
# ERA5 file
era5_file = './era5_2020_06_12.nc

# FVCOM grid configuration from existing restart file, could also create from .dat files using grid.createg_grid
sval_res = pf.FVCOMReader(f'/Sval_restart_20170301.nc')
fvcom_grid = sval_res.grid

# Define the simulation time period
start_date_time = datetime.strptime("20200701", "%Y%m%d")
end_date_time = datetime.strptime("20200801", "%Y%m%d%H")

# Create array of hourly datetime objects
date_times = pf.date_utils.create_datetime_array(start_date_time, end_date_time, timedelta(hours=1))


CMEMS data directory: /users/modellers/jcl/data/pyfvcom2_doc/CMEMS_NWS_anfc_0.027deg
Grid file: /users/modellers/jcl/data/pyfvcom2_doc/FVCOM_tamar_estuary/tamar_v2_grd.dat
Open boundary file: /users/modellers/jcl/data/pyfvcom2_doc/FVCOM_tamar_estuary/tamar_v2_obc.dat
Sigma file: /users/modellers/jcl/data/pyfvcom2_doc/FVCOM_tamar_estuary/sigma_gen.dat


## 3. Use ERA5Reader, ERA5Interpolater and SurfaceManager to write interpolate the data and create a surface forcing file

Create the Reader, Interpolation, and Manager objects which respectively; read the ERA5 data and perform necessary variable conversions, interpolate the data between grids, and manage writing the netcdf and getting data from different sources if required.

In [5]:
era5_reader = pf.surface.ERA5Reader(era5_file, reference_var_name='u10')
era5_interpolator = pf.surface.ERA5Interpolator(era5_reader)

surface_manager =  pf.surface.SurfaceManager(fvcom_grid)
surface_manager.set_dates(date_times)

# This loops over all the default atmospher
surface_manager.add_standard_forcing(era5_interpolator)

surface_manager.create_forcing_file('fvcom_atm_2020_07.nc')


Updating NestManager dates and purging old forcing data for the previous dates.
NestManager created successfully!
Number of grid bands: 2
Interpolation method: linear


## 4. Use SurfaceManager to combine the ERA5 atmospheric data with CMEMS ice data to create a surface forcing file

Here we will repeat the process, but also download data from the CMEMS topaz model and add ice cover data from a second reader to the SurfaceManager as well (required if for example using the ICENUDGE module in FVCOM)

In [ ]:
import copernicusmarine

# Daily-mean dataset inside the product (the main TOPAZ4b daily output)
DATASET_ID = "cmems_mod_arc_phy_my_topaz4_P1D-m"
variables = [
    "siconc",    # Sea ice area fraction
    "sithick",   # Sea ice thickness
    "so",        # Sea water salinity
    "thetao",    # Sea water potential temperature
    "vxo",        # Eastward sea water velocity
    "vyo",        # Northward sea water velocity
    "zos",       # Sea surface height above geoid
    "vxsi",       # Sea ice eastward velocity
    "vysi",       # Sea ice northward velocity
]

# Call login with credentials if needed
#copernicusmarine.login(**login_kwargs, skip_if_user_logged_in=True)

# Download via the Copernicus Marine Toolbox subset API
copernicusmarine.subset(
        dataset_id=DATASET_ID,
        variables=variables,
        start_datetime=start_date_time,
        end_datetime=end_date_time,
        minimum_longitude=lon_min,
        maximum_longitude=lon_max,
        minimum_latitude=lat_min,
        maximum_latitude=lat_max,
        output_filename='./topaz_2020_06',
        force_download=True,
    )


In [6]:
# Adding the era5 data is as befor
era5_reader = pf.surface.ERA5Reader(force_file, 'u10')
era5_interpolator = pf.surface.ERA5Interpolator(era5_reader)

surface_manager =  pf.surface.SurfaceManager(fvcom_grid)
surface_manager.set_dates(date_times)
surface_manager.add_standard_forcing(era5_interpolator)

# But now also add the Topaz reader/interpolator
topaz_file = '/topaz_2020_06.nc'
topaz_reader = pf.forcing_reader.CMEMSReader(topaz_file, reference_var_name='so')
topaz_var_names = {'HICE':'sithick', 'AICE':'siconc', 'ISALT':'siconc', 'UICE':'vxsi', 'VICE':'vysi'}
topaz_interpolator = pf.interpolation.CMEMSInterpolator(topaz_reader, fvcom_to_cmems_var_names=topaz_var_names)

for fvcom_var in ['AICE', 'HICE', 'ISALT']:
    surface_manager.add_forcing_data(topaz_interpolator, fvcom_var, horizontal_position='node')

for fvcom_var in ['UICE', 'VICE']:
    surface_manager.add_forcing_data(topaz_interpolator, fvcom_var, horizontal_position='element')

# And make the forcing file, the ice variable conversion is already given in pyfvcom, but variables of any type
# could be passed, provided as a dictionary of 'node' and 'element' variables with their variable attributes
#default_ice_nudge_vars = {'node':
#    {'AICE':{'long_name':'Ice Cover', 'units':'fraction [0-1]'},
#     'HICE':{'long_name':'Ice Thickness', 'units':'[m]'},
#     'ISALT':{'long_name':'Ice salinity', 'units':''}},
#   'element':
#    {'UICE':{'long_name':'Eastward Ice Speed', 'units':'m/s'},
#     'VICE':{'long_name':'Northward Ice Speed', 'units':'m/s'}}}


surface_manager.create_forcing_file('fvcom_atm_and_ice_2020_07.nc', additional_surface_variables=pf.surface.default_ice_nudge_vars))

Found 1 2D CMEMS files:
  1: /users/modellers/jcl/data/pyfvcom2_doc/CMEMS_NWS_anfc_0.027deg/cmems_mod_nws_phy_anfc_0.027deg-2D_PT1H-m/cmems_mod_nws_phy_anfc_0.027deg-2D_PT1H-m_20250614_20250614.nc
